In [2]:
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field

texts = []
for path in sorted(Path("data").glob("student_text_*.txt")):
    texts.append(path.read_text(encoding="utf-8"))

len(texts)

2

In [3]:
from pydantic_ai import Agent
from model import nvidia_model

model = nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b")

class Student(BaseModel):
    proposed_grade: str = Field(description="Grade this based on the text from range A-F with a being the highest grade, and F lowest grade")
    motivation: str = Field(description="Motivate why the student deserve this grade in 2-3 sentances")
    improvements: str = Field(description="Suggest what the student could improve in 1-2 sentances")

grading_agent = Agent(
    model=model,
    instructions=("Du är en erfaren svensklärare på gymnasiet. "
                "Du bedömer elevtexter och ger ett betygförslag. Svara på svenska. "
    ),
)

async def grade_text(text: str) -> Student:
    result = await grading_agent.run(text, output_type=Student)
    return result.output

In [5]:
for i, text in enumerate(texts, start=1):
    grade = await grade_text(text)

    folder = Path("grades") / f"student_{i}"
    folder.mkdir(parents=True, exist_ok=True)

    (folder / "proposed_grade.txt").write_text(grade.proposed_grade, encoding="utf-8")
    (folder / "motivation.txt").write_text(grade.motivation, encoding="utf-8")
    (folder / "improvements.txt").write_text(grade.improvements, encoding="utf-8")

In [9]:
all_text = Path("data/svenska1_criterias.txt").read_text(encoding="utf-8")

start = all_text.find("Betygskriterier")
criteria = all_text[start:]

print(criteria)


Betygskriterier
Betyget E
Eleven kan, i förberedda samtal och diskussioner, muntligt förmedla egna tankar och åsikter samt genomföra muntlig framställning inför en grupp. Detta gör eleven med viss säkerhet. Den muntliga framställningen är sammanhängande och begriplig. Eleven har viss åhörarkontakt. Språket är till viss del anpassat till syfte, mottagare och kommunikations­situation. Vidare kan eleven med viss säkerhet använda presentationstekniska hjälpmedel.

Eleven kan skriva argumenterande text och andra typer av texter, som är sammanhängande och begripliga samt till viss del anpassade till syfte, mottagare och kommunikations­situation. Eleven kan i huvudsak följa skriftspråkets normer för språkriktighet.

Eleven kan läsa, reflektera över och göra enkla sammanfattningar av texter samt skriva egna texter som anknyter till det lästa. I sitt arbete värderar och granskar eleven med viss säkerhet källor kritiskt samt tillämpar grundläggande regler för citat- och referatteknik.

Eleven ka

In [10]:
criteria_agent = Agent(
    model=model,
    instructions=(
        "Du är en erfaren svensklärare på gymnasiet"
        "Du bedömer elevtexter och ger betygsförslag. Svara på svenska"
        "Bedöm texten enbart utifrån följande betygskriterier:\n\n"
        + criteria
    ),
)

async def grade_with_criteria(text: str) -> Student:
    result = await criteria_agent.run(text, output_type=Student)
    return result.output

In [12]:
for i, text in enumerate(texts, start=1):
    without = await grade_text(text)
    with_criteria = await grade_with_criteria(text)

print(f"Elev {i}")
print(" utan kriterier:", without.proposed_grade)
print(" med kriterier:", with_criteria.proposed_grade)


Elev 2
 utan kriterier: C
 med kriterier: E
